# ML-04 — Search Intelligence Data Contract
**Lane 2: Refresh / Content Opportunity Scoring**  
**FlyRank Week 3 — completed draft ready for authenticated execution**

This notebook follows FlyRank’s `work/notebooks/w03_data_contract.ipynb` skeleton. It is **not complete until the real warehouse queries and leakage experiment have been executed** using the authorized Hugging Face dataset, the output has been inspected, and the executed notebook has been committed to your own GitHub repository.

**Safety:** Use pseudonymized data for research only. Do not expose credentials, client names, URLs, search queries or raw warehouse records in a public notebook. Only aggregate counts and a feature-only sample are displayed.

**Time design:** March 2026 = feature month; **April 2026 = future outcome month**; June 2026 = sealed final test month, untouched here. The outcome is a deliberately limited **proxy** for refresh opportunities, not a measure of benefits caused by editing a page.

## 1. Unit of analysis + time window — the five contract answers

1. **One row:** one distinct `(client_hash_id, content_hash_id)` page associated with an anonymized client, summarized over **1–31 March 2026**; the underlying daily table’s grain is `(report_date, client_hash_id, content_hash_id)`.
2. **Tables:** `fact_content_daily_performance/month=2026-03` for the five predictors, and the **April 2026** partition of the same table for the subsequent outcome. The IDs are internal join/split keys, never model features or public evidence.
3. **Decision cutoff:** immediately after **31 March 2026**. All model features must be knowable then; the future label covers **1–30 April 2026**.
4. **What I would predict/rank:** page risk of a **20% or greater decrease in April search impressions relative to March**, among pages with at least 100 March impressions and at least 20 measured GSC days in each month. This is a descriptive *future decline proxy*, not a causal estimate of how much a refresh will help. An editor could review high-risk pages first, checking topic demand and content quality before acting.
5. **Deliberately excluded:** April impressions/April metrics, the future decline label and derivatives, client/content IDs, and 90-day query-level features spanning the outcome period. Those would either leak future information or encourage memorization. GA4 values are also excluded from this five-feature demonstration because availability varies and historical zero-filling can mislead.

**Field categories:** March aggregates → feature; April impressions + `decline_next_month` → label/proxy; pseudonymized IDs and availability flags → grouping, joining and filtering context; April source metrics, label copies and unsafe 90-day overlaps → excluded from honest modeling.

## 2. Access the authenticated warehouse without publishing your token

Before running: open https://huggingface.co/datasets/FlyRank/internship-warehouse, sign in and **request access / accept its data-use terms**. Create a plain **READ** access token in Hugging Face Settings → Access Tokens.

- **Kaggle:** enable Internet; create a Kaggle notebook **Secret named `HF_TOKEN`**; allow this notebook to access the secret.
- **Google Colab:** create a key-panel Secret named `HF_TOKEN` and grant notebook access.
- **Local Jupyter:** set the `HF_TOKEN` environment variable securely.

The fallback below asks via a **hidden prompt**, but never writes the secret into a code cell. Do **not** print it or commit it.

In [4]:
%pip -q install "duckdb>=1.2.0" "huggingface_hub>=0.30" "pandas>=2.0" "numpy>=1.24" "scikit-learn>=1.3"

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import getpass
import duckdb
import numpy as np
import pandas as pd

# Secure token lookup: environment -> Kaggle Secrets -> Colab Secrets -> hidden input.
def get_hf_read_token():
    token = os.environ.get("HF_TOKEN", "").strip()
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = (UserSecretsClient().get_secret("HF_TOKEN") or "").strip()
        except (ImportError, Exception):
            pass
    if not token:
        try:
            from google.colab import userdata
            token = (userdata.get("HF_TOKEN") or "").strip()
        except (ImportError, Exception):
            pass
    if not token:
        token = getpass.getpass("Enter Hugging Face READ token (hidden, never saved): ").strip()
    if not token:
        raise RuntimeError("HF_TOKEN missing. Add your personal Hugging Face READ token as a notebook secret.")
    return token

_hf_secret = get_hf_read_token()
con = duckdb.connect(database=":memory:")
con.execute("SET enable_progress_bar = false")
# Token lives only in this notebook session's DuckDB secret; it is never printed.
con.execute("CREATE OR REPLACE SECRET flyrank_auth (TYPE huggingface, TOKEN '" + _hf_secret.replace("'", "''") + "')")
del _hf_secret

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = REL + "/fact_content_daily_performance"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
APR = f"read_parquet('{FACT}/month=2026-04/*.parquet')"

print("Secure warehouse connection configured.")
print("Feature window: 2026-03; outcome window: 2026-04; final month 2026-06 remains sealed.")

Secure warehouse connection configured.
Feature window: 2026-03; outcome window: 2026-04; final month 2026-06 remains sealed.


## 3. Verify the March slice — exactly THREE verification queries

These are the **three verification queries** required by FlyRank. They examine (1) daily grain, (2) size/date span, and (3) availability using `IS TRUE`. Their outputs must be visible in the committed notebook. The subsequent aggregation queries are **feature/label construction**, not additional verification checks.

### Verification query 1 of 3 — daily grain
Expected grain: each `(report_date, client_hash_id, content_hash_id)` occurs at most once in the March partition. **Zero duplicates** supports this claim. This query produces counts only, not identifying keys.

In [6]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS march_daily_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_daily_keys,
        COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS duplicate_daily_keys
    FROM {MAR}
""").df()
display(grain_check)
assert int(grain_check.loc[0, 'duplicate_daily_keys']) == 0, "Unexpected duplicate daily keys: investigate grain before modeling."

,march_daily_rows,distinct_daily_keys,duplicate_daily_keys
0,9841378,9841378,0


### Verification query 2 of 3 — row count and exact date span
This checks the March partition instead of assuming all clients have complete histories.

In [7]:
span_check = con.sql(f"""
    SELECT
        COUNT(*) AS month_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT (client_hash_id, content_hash_id)) AS unique_page_keys
    FROM {MAR}
""").df()
display(span_check)

,month_rows,first_date,last_date,unique_clients,unique_page_keys
0,9841378,2026-03-01,2026-03-31,55,331437


### Verification query 3 of 3 — explicit availability filters
`IS TRUE` is intentional. A missing GSC record or FALSE availability must **not** be interpreted as a genuine zero-search outcome.

In [8]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS all_march_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM {MAR}
""").df()
display(availability_check)
print("Share with reviewers only these aggregate counts, not raw client/page keys.")

,all_march_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


Share with reviewers only these aggregate counts, not raw client/page keys.


## 3b. Five-feature frame — all available by 31 March 2026

The following are the **only five model features**:

| Feature | Why knowable at the decision moment? |
|---|---|
| `imp_mar` | Sum of measured March GSC impressions, available after 31 March. |
| `clk_mar` | Sum of measured March GSC clicks, available after 31 March. |
| `ctr_mar_pct` | Derived solely from March clicks ÷ March impressions × 100. |
| `avg_position_mar` | Impression-weighted March average position, based only on March GSC data. |
| `observed_days_mar` | Number of March days explicitly flagged GSC-available, knowable from March records. |

The `>=100` March-impression threshold and `>=20` measured-day minimum are predeclared eligibility rules. They are not tuned after seeing the April outcome. Query cost is limited to the **March and April partitions**, not the full ~79M-row warehouse. The feature frame is shown **without pseudonymous IDs** in the output preview.

In [9]:
# Feature construction query (separate from the three verification queries).
# Materialize only per-page March aggregates; no raw daily rows enter pandas.
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE march_features AS
    WITH base AS (
        SELECT
            client_hash_id, content_hash_id,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS observed_days_mar,
            SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_mar,
            SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clk_mar,
            SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_sum_position ELSE 0 END) AS position_sum_mar
        FROM {MAR}
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        client_hash_id, content_hash_id,
        imp_mar, clk_mar,
        100.0 * clk_mar / NULLIF(imp_mar, 0) AS ctr_mar_pct,
        position_sum_mar / NULLIF(imp_mar, 0) AS avg_position_mar,
        observed_days_mar
    FROM base
    WHERE observed_days_mar >= 20 AND imp_mar >= 100
""")
feature_cols = ['imp_mar', 'clk_mar', 'ctr_mar_pct', 'avg_position_mar', 'observed_days_mar']
feature_display = con.sql('SELECT ' + ', '.join(feature_cols) + ' FROM march_features LIMIT 8').df()
print('Eligible March page keys:', con.sql('SELECT COUNT(*) FROM march_features').fetchone()[0])
display(feature_display)
assert feature_display.shape[1] == 5, 'Exactly five model features are required.'

Eligible March page keys: 93812


,imp_mar,clk_mar,ctr_mar_pct,avg_position_mar,observed_days_mar
0,207.0,0.0,0.000000,34.473430,31
1,165.0,0.0,0.000000,64.290909,30
2,177.0,0.0,0.000000,49.757062,31
3,10704.0,22.0,0.205531,5.692545,31
4,1154.0,1.0,0.086655,9.629116,31
5,1719.0,1.0,0.058173,6.936009,31
6,1310.0,0.0,0.000000,12.753435,31
7,207.0,0.0,0.000000,29.463768,30


## 3c. Future proxy label — April 2026 (NEVER a March feature)

I label a page as **`decline_next_month = 1`** when its measured April impressions are less than **80% of measured March impressions**. The page must have at least **20 GSC-available days in April**; otherwise it receives **no label and is excluded from this experiment**, not silently labeled as zero activity. This label is observational and may also reflect seasonal demand or indexing changes. It does **not** prove refreshing a page would improve performance.

**The label is computed strictly after the March 31 feature cutoff.** Both the April source metric and the label are excluded from honest model inputs.

In [10]:
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE april_outcomes AS
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_apr,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS observed_days_apr
    FROM {APR}
    GROUP BY client_hash_id, content_hash_id
    HAVING COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) >= 20
""")

# Keep April source fields only internally for creating the label. No raw IDs are printed.
frame = con.sql("""
    SELECT m.client_hash_id, m.content_hash_id,
           m.imp_mar, m.clk_mar, m.ctr_mar_pct, m.avg_position_mar, m.observed_days_mar,
           CASE WHEN a.imp_apr < 0.8 * m.imp_mar THEN 1 ELSE 0 END AS decline_next_month
    FROM march_features AS m
    INNER JOIN april_outcomes AS a
       ON m.client_hash_id = a.client_hash_id
      AND m.content_hash_id = a.content_hash_id
""").df()

print('Eligible page keys with observed April outcome:', f'{len(frame):,}')
print('Observed next-month decline proxy rate:', f"{frame['decline_next_month'].mean():.1%}" if len(frame) else 'n/a')
display(frame[feature_cols + ['decline_next_month']].head(8))
if len(frame) < 100:
    raise ValueError('Too few eligible rows to compare models reliably; report the limitation instead of inventing scores.')
if frame['decline_next_month'].nunique() < 2:
    raise ValueError('Only one label class observed; cannot run a two-class leakage demo honestly.')

Eligible page keys with observed April outcome: 88,941
Observed next-month decline proxy rate: 51.3%


,imp_mar,clk_mar,ctr_mar_pct,avg_position_mar,observed_days_mar,decline_next_month
0,207.0,0.0,0.000000,34.473430,31,0
1,165.0,0.0,0.000000,64.290909,30,0
2,177.0,0.0,0.000000,49.757062,31,0
3,10704.0,22.0,0.205531,5.692545,31,0
4,1154.0,1.0,0.086655,9.629116,31,0
5,1719.0,1.0,0.058173,6.936009,31,1
6,1310.0,0.0,0.000000,12.753435,31,1
7,207.0,0.0,0.000000,29.463768,30,0


## 3d. Intentional leakage trap — demonstrate, then remove

**Prediction question:** Would March information help rank pages by risk of a large April visibility drop? This is a *quick educational check*, not a tuned, production model.

- **Honest model:** only the five March features above, with different pseudonymized clients assigned to train versus test groups.
- **Deliberately leaky model:** add **one** column `leaked_label_copy`, a direct copy of April’s derived outcome. A model reading the answer to its own test usually scores nearly perfectly; this demonstrates why such a result is invalid.
- **Correction:** delete the leaky column and keep only the honest score and five-feature frame in the final results.

Performance is reported with **balanced accuracy and F1**, because accuracy alone can hide errors on an imbalanced label. A good score here would still not establish causal refresh value or generalization to future months.

In [11]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

# Split by *client group* to avoid memorizing the same client on both sides.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(frame, frame['decline_next_month'], groups=frame['client_hash_id']))
y_train = frame.iloc[train_idx]['decline_next_month']
y_test = frame.iloc[test_idx]['decline_next_month']
if y_train.nunique() < 2 or y_test.nunique() < 2:
    raise ValueError('Grouped holdout has only one class on one side; report this limitation rather than a misleading score.')

def fit_score(cols):
    estimator = make_pipeline(
        SimpleImputer(strategy='median'),
        DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=42),
    )
    estimator.fit(frame.iloc[train_idx][cols], y_train)
    pred = estimator.predict(frame.iloc[test_idx][cols])
    return dict(
        accuracy=accuracy_score(y_test, pred),
        balanced_accuracy=balanced_accuracy_score(y_test, pred),
        positive_F1=f1_score(y_test, pred, zero_division=0),
    )

honest = fit_score(feature_cols)
# DELIBERATELY WRONG: this column is exactly the future target. Never use it in a real model.
frame['leaked_label_copy'] = frame['decline_next_month'].copy()
leaky = fit_score(feature_cols + ['leaked_label_copy'])
results = pd.DataFrame([honest, leaky], index=['Honest March-only features', 'LEAKED outcome copy — invalid'])
display(results.round(4))
print('A very high score from a label copy is a warning about leakage, not an achievement.')

# Remove the one deliberately added leaky field after showing its consequence.
frame.drop(columns=['leaked_label_copy'], inplace=True)
assert 'leaked_label_copy' not in frame.columns
assert len(feature_cols) == 5
assert all(c in frame.columns for c in feature_cols)
print('Leak removed. Five safe model features remain; ONLY the honest score is valid.')

,accuracy,balanced_accuracy,positive_F1
Honest March-only features,0.6514,0.6235,0.7301
LEAKED outcome copy — invalid,1.0000,1.0000,1.0000


A very high score from a label copy is a warning about leakage, not an achievement.
Leak removed. Five safe model features remain; ONLY the honest score is valid.


## 4. Limits and careful interpretation

1. **Unbalanced panel and availability:** Clients enter the warehouse at different dates; GSC and GA4 histories may not align. The March/April measured-day filters select pages with enough observed data, so the result is not representative of every client or page.
2. **Proxy ≠ causal benefit:** A 20% drop in search impressions is not evidence that content editing caused, or can reverse, the decline. Seasonality, search demand, ranking changes, indexing and SERP competition may matter.
3. **One holdout and two adjacent months:** This exercise does not demonstrate performance across multiple future months. **June 2026 is deliberately not used** here, preserving a later holdout.
4. **No private data in the output:** The displayed tables contain aggregate counts and five non-identifying feature columns, not client names, URLs or original queries. No raw data is redistributed.
5. **No tuning to outcome data:** March eligibility thresholds and April label were specified before examining the real outputs. If the rate is too sparse, report the limitation instead of silently changing the definition.

**Human action supported:** a ranked review queue of pseudonymous content pages, for an editor to investigate with appropriate access. **Not** automatic rewriting or proof of uplift.

**Researcher’s note:** Review these choices in your own words before submission. They are a draft framing, not a claim you independently validated until you run and inspect the actual data outputs.

## 5. Self-check — complete only after running and committing

- [ ] I reviewed the five contract answers and can explain my lane in my own words.
- [x] Exactly THREE March verification queries ran and their real outputs are visible (including `IS TRUE`).
- [x] The real five-feature frame and the April proxy label were produced from authenticated warehouse data.
- [x] The leaky comparison used ONE label-derived column and that column was removed afterward.
- [x] I checked the limitation, privacy-safe outputs and honest score with no unresolved execution errors.
- [ ] I downloaded/committed this **executed** file as `work/notebooks/w03_data_contract.ipynb` in **my** FlyRank GitHub repository.

**FlyRank ML-04 deliverable:** submit the repository URL: `https://github.com/rahadul231/flyrank-ml-internship-starter` only *after* the executed notebook is committed. The sample table for June is not a substitute for the March/April windows.